# Shallow LightGBM Experiments

The best model so far is `LightGBM` with 4 leaves and depth 2 (OOF ROC-AUC **0.947895**).

Trees this shallow cannot build interactions on their own, so earlier feature engineering results may not hold anymore. We will:

1. measure how much the score changes between different CV seeds,
2. repeat the feature experiments on the shallow model,
3. look for pairs of features that actually interact.

Every experiment is compared with the baseline on the same CV splits.

In [ ]:
import itertools
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

root = Path("../playground-series-s6e11")
train_df = pd.read_csv(root / "train.csv")

y = train_df["is_fraud"].astype(int)
X = train_df.drop(columns=["id", "is_fraud"]).copy()

for col in X.select_dtypes(include="bool").columns:
    X[col] = X[col].astype(int)

cat_cols = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
for col in cat_cols:
    X[col] = X[col].astype("category")

base_features = X.columns.tolist()
print(X.shape, cat_cols)

(699960, 24) ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']


In [ ]:
lgb_params = {
    "n_estimators": 5000,
    "learning_rate": 0.05,
    "objective": "binary",
    "metric": "auc",
    "num_leaves": 4,
    "max_depth": 2,
    "min_child_samples": 100,
    "reg_lambda": 10,
    "verbosity": -1,
    "n_jobs": -1,
}


def run_cv(df, features, seed, params=None):
    params = {**lgb_params, **(params or {})}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    oof = np.zeros(len(df))
    best_iters = []

    for train_idx, valid_idx in cv.split(df, y):
        model = lgb.LGBMClassifier(**params, random_state=seed)

        model.fit(
            df.iloc[train_idx][features],
            y.iloc[train_idx],
            eval_set=[(df.iloc[valid_idx][features], y.iloc[valid_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(150, verbose=False)],
        )

        oof[valid_idx] = model.predict_proba(df.iloc[valid_idx][features])[:, 1]
        best_iters.append(model.best_iteration_)

    return roc_auc_score(y, oof), np.mean(best_iters)

## Seed Noise

We run the baseline with several different CV seeds. The spread between them tells us how large a difference has to be before we can trust it.

In [ ]:
seeds = [42, 43, 44]

baseline_scores = []

for seed in seeds:
    auc, iters = run_cv(X, base_features, seed)
    baseline_scores.append(auc)
    print(f"Seed {seed}: AUC={auc:.6f}, avg_best_iteration={iters:.0f}")

baseline_scores = np.array(baseline_scores)

print(f"\nMean: {baseline_scores.mean():.6f}")
print(f"Std:  {baseline_scores.std(ddof=1):.6f}")
print(f"Max - min: {np.ptp(baseline_scores):.6f}")

Seed 42: AUC=0.947895, avg_best_iteration=2708
Seed 43: AUC=0.947710, avg_best_iteration=2544
Seed 44: AUC=0.947890, avg_best_iteration=2783

Mean: 0.947832
Std:  0.000106
Max - min: 0.000186


## Seed Stability
The baseline scored ***0.947895***, ***0.947710***, and ***0.947890*** for seeds 42, 43, and 44, respectively (mean ***0.947832***, standard deviation ***0.000106***).

This spread gives useful context for small score changes, although three seeds are not enough to establish a formal uncertainty interval. We will compare feature changes on the same splits rather than ranking tiny absolute differences.

## Feature Engineering on a Shallow Model

For each experiment we run the same seeds as the baseline and look at the difference per seed.

A monotonic transformation of a single feature (for example `log(amount)`) does not change a tree model, so we only test features that combine several columns.

In [ ]:
results = []


def evaluate(name, df, features):
    scores = np.array([run_cv(df, features, seed)[0] for seed in seeds])
    delta = scores - baseline_scores

    results.append({
        "experiment": name,
        "oof_auc": scores.mean(),
        "delta": delta.mean(),
        "delta_std": delta.std(ddof=1),
        "better_in_seeds": f"{(delta > 0).sum()}/{len(seeds)}",
    })

    print(f"{name}: AUC={scores.mean():.6f}, delta={delta.mean():+.6f}")

In [ ]:
new_features = {
    "risk_velocity": lambda d: d["merchant_risk_score"] * d["velocity_score"],
    "txn_frequency": lambda d: d["txn_count_last_24h"] / (d["hours_since_last_txn"] + 1),
    "amount_to_balance_ratio": lambda d: d["amount_usd"] / d["account_balance_usd"].replace(0, np.nan),
    "log_amount_x_risk": lambda d: np.log1p(d["amount_usd"]) * d["merchant_risk_score"],
}

for name, fn in new_features.items():
    experiment_df = X.copy()
    experiment_df[name] = fn(experiment_df)

    evaluate(f"add {name}", experiment_df, base_features + [name])

add risk_velocity: AUC=0.947609, delta=-0.000223
add txn_frequency: AUC=0.947678, delta=-0.000154
add amount_to_balance_ratio: AUC=0.947701, delta=-0.000131
add log_amount_x_risk: AUC=0.947674, delta=-0.000157


### Removing noisy features

During EDA these features showed almost no relationship with fraud. Removing them one by one changed the score by about 0.0002, so here we remove the whole group at once.

In [ ]:
noise_features = [
    "distance_from_home_km",
    "card_age_months",
    "time_of_day_hour",
    "day_of_week",
    "device_type",
    "channel",
]

evaluate(
    "drop noise group",
    X,
    [c for c in base_features if c not in noise_features],
)

pd.DataFrame(results).round(6)

drop noise group: AUC=0.947827, delta=-0.000005


,experiment,oof_auc,delta,delta_std,better_in_seeds
0,add risk_velocity,0.947609,-0.000223,0.000023,0/3
1,add txn_frequency,0.947678,-0.000154,0.000099,0/3
2,add amount_to_balance_ratio,0.947701,-0.000131,0.000034,0/3
3,add log_amount_x_risk,0.947674,-0.000157,0.000036,0/3
4,drop noise group,0.947827,-0.000005,0.000070,1/3


## Feature Engineering Findings
None of the tested arithmetic features improved the shallow `LightGBM` across the three seeds. Their mean AUC changes ranged from ***−0.000131*** to ***−0.000223***.

Removing the group of weak features made almost no difference (***−0.000005*** on average; better on only one of three seeds). We will keep the original feature set.

## Depth 1 vs Depth 2

In the `CatBoost` experiments depth 1 used all 8000 iterations, so it did not converge and the comparison with depth 2 was not fair.

Here we use a higher learning rate so every model can finish training.

In [ ]:
depth_configs = {
    "depth1": {"num_leaves": 2, "max_depth": 1},
    "depth2": {"num_leaves": 4, "max_depth": 2},
    "depth3": {"num_leaves": 8, "max_depth": 3},
}

depth_results = []

for name, config in depth_configs.items():
    auc, iters = run_cv(X, base_features, 42, {"learning_rate": 0.2, **config})
    depth_results.append({"model": name, "oof_auc": auc, "avg_best_iteration": iters})

pd.DataFrame(depth_results).round(6)

,model,oof_auc,avg_best_iteration
0,depth1,0.947619,3587.6
1,depth2,0.947728,811.0
2,depth3,0.946102,355.8


## Tree Depth Findings
In this separate depth comparison, depth 2 achieved ***0.947728***, depth 1 ***0.947619***, and depth 3 ***0.946102***.

The smaller trees again performed well. However, this experiment uses a different learning rate from the main baseline, so these results should not be treated as a direct replacement for the baseline score of ***0.947895***.

## Looking for Interactions

### EBM

`ExplainableBoostingClassifier` searches for pairwise interaction terms by itself. It is slow on the full data, so we train it on a 200k sample.

In [ ]:
from interpret.glassbox import ExplainableBoostingClassifier

sample_idx = X.sample(200_000, random_state=0).index

X_ebm = X.loc[sample_idx].copy()
for col in cat_cols:
    X_ebm[col] = X_ebm[col].astype(str)

ebm = ExplainableBoostingClassifier(interactions=20, random_state=42, n_jobs=-1)
ebm.fit(X_ebm, y.loc[sample_idx])

terms = pd.DataFrame({
    "term": ebm.term_names_,
    "importance": ebm.term_importances(),
}).sort_values("importance", ascending=False)

is_pair = terms["term"].str.contains(" & ", regex=False)

terms[~is_pair].head(10)

,term,importance
11,merchant_risk_score,0.967315
8,velocity_score,0.887442
5,customer_age,0.414314
15,auth_method,0.408230
19,is_new_merchant,0.386690
7,cvv_retry_count,0.361013
1,hours_since_last_txn,0.195877
13,merchant_category,0.160444
2,txn_count_last_24h,0.127507
20,used_vpn,0.101650


In [ ]:
terms[is_pair].head(10)

,term,importance
38,customer_age & merchant_risk_score,0.044058
39,velocity_score & merchant_risk_score,0.041352
25,amount_usd & merchant_risk_score,0.034447
27,hours_since_last_txn & merchant_risk_score,0.029995
33,distance_from_home_km & merchant_risk_score,0.028196
24,amount_usd & customer_age,0.022977
37,card_age_months & merchant_risk_score,0.016832
31,txn_count_last_24h & merchant_risk_score,0.015571
30,txn_count_last_24h & velocity_score,0.015421
26,hours_since_last_txn & account_balance_usd,0.014717


### Pair scan with LightGBM

As a second opinion we fit an additive `LightGBM` where every tree can use only one feature, and then allow one pair of features at a time to interact. A pair that raises validation AUC over the additive model is a candidate.

In [ ]:
scan_idx = X.sample(250_000, random_state=1).index

X_a, X_b, y_a, y_b = train_test_split(
    X.loc[scan_idx],
    y.loc[scan_idx],
    test_size=0.3,
    stratify=y.loc[scan_idx],
    random_state=0,
)

col_pos = {c: i for i, c in enumerate(base_features)}

scan_params = {
    "n_estimators": 3000,
    "learning_rate": 0.1,
    "num_leaves": 4,
    "max_depth": 2,
    "min_child_samples": 100,
    "reg_lambda": 10,
    "verbosity": -1,
    "n_jobs": -1,
    "random_state": 0,
}


def fit_with_groups(groups):
    model = lgb.LGBMClassifier(
        **scan_params,
        interaction_constraints=[[col_pos[c] for c in g] for g in groups],
    )
    model.fit(
        X_a, y_a,
        eval_set=[(X_b, y_b)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    return roc_auc_score(y_b, model.predict_proba(X_b)[:, 1]), model


single_groups = [[c] for c in base_features]
additive_auc, additive_model = fit_with_groups(single_groups)
print(f"Additive model AUC: {additive_auc:.6f}")

Additive model AUC: 0.944615


In [ ]:
gain = pd.Series(
    additive_model.booster_.feature_importance("gain"),
    index=base_features,
)

top_features = [c for c in gain.sort_values(ascending=False).index if c not in cat_cols][:10]
print(top_features)

pair_rows = []

for a, b in itertools.combinations(top_features, 2):
    groups = [g for g in single_groups if g[0] not in (a, b)] + [[a, b]]
    auc, _ = fit_with_groups(groups)
    pair_rows.append({"feature_a": a, "feature_b": b, "auc_gain": auc - additive_auc})

pair_scan = pd.DataFrame(pair_rows).sort_values("auc_gain", ascending=False)
pair_scan.head(15).round(6)

['cvv_retry_count', 'merchant_risk_score', 'velocity_score', 'is_new_merchant', 'ip_country_mismatch', 'billing_shipping_mismatch', 'customer_age', 'used_vpn', 'is_foreign_transaction', 'is_ai_generated_scam_attempt']


,feature_a,feature_b,auc_gain
9,merchant_risk_score,velocity_score,0.000213
15,merchant_risk_score,is_foreign_transaction,0.000121
4,cvv_retry_count,billing_shipping_mismatch,0.000067
38,billing_shipping_mismatch,is_ai_generated_scam_attempt,0.000039
22,velocity_score,is_foreign_transaction,0.000038
8,cvv_retry_count,is_ai_generated_scam_attempt,0.000032
34,ip_country_mismatch,is_ai_generated_scam_attempt,0.000027
27,is_new_merchant,used_vpn,0.000025
21,velocity_score,used_vpn,0.000024
29,is_new_merchant,is_ai_generated_scam_attempt,0.000023


## Adding Interacting Pairs as Features

A single validation split is noisy, so the best pairs from both methods are checked with the full CV.

In [ ]:
pairs = [
    ("merchant_risk_score", "velocity_score"),
]

pairs_df = X.copy()
pair_features = []

for a, b in pairs:
    pairs_df[f"{a}_x_{b}"] = pairs_df[a] * pairs_df[b]
    pairs_df[f"{a}_div_{b}"] = pairs_df[a] / (pairs_df[b].abs() + 1e-3)
    pair_features += [f"{a}_x_{b}", f"{a}_div_{b}"]

evaluate("add pairs", pairs_df, base_features + pair_features)

pd.DataFrame(results).round(6)

add pairs: AUC=0.947547, delta=-0.000285


,experiment,oof_auc,delta,delta_std,better_in_seeds
0,add risk_velocity,0.947609,-0.000223,0.000023,0/3
1,add txn_frequency,0.947678,-0.000154,0.000099,0/3
2,add amount_to_balance_ratio,0.947701,-0.000131,0.000034,0/3
3,add log_amount_x_risk,0.947674,-0.000157,0.000036,0/3
4,drop noise group,0.947827,-0.000005,0.000070,1/3
5,add pairs,0.947547,-0.000285,0.000028,0/3


## Final Findings
The shallow `LightGBM` remained our strongest individual model from the earlier tuning stage (***0.947895*** OOF ROC-AUC at seed 42).

In this follow-up, none of the tested feature additions improved its mean AUC across the three CV seeds. Adding features based on `merchant_risk_score` and `velocity_score` achieved ***0.947547***, a mean change of ***−0.000285*** versus the corresponding baseline. Removing a group of weak features also provided no meaningful gain.

The interaction scans identified plausible candidate pairs, but those relationships did not translate into a better classifier in the experiment we ran.

**Conclusion**: We will keep the original features and the tuned shallow LightGBM. More handcrafted interactions are unlikely to be the best use of time, so we will stop feature engineering here.